<a href="https://colab.research.google.com/github/Chosencodes/Medical-Imaging-Projects/blob/main/Pneumonia_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install kaggle

In [ ]:
from google.colab import files
files.upload()

In [ ]:
!mkdir ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!pip install pydicom

In [ ]:
import numpy as np
import pandas as pd
import cv2
import pydicom
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm.notebook import tqdm

In [ ]:
!kaggle competitions download -c rsna-pneumonia-detection-challenge

In [ ]:
!unzip rsna-pneumonia-detection-challenge.zip

In [ ]:
# !pip install pytorch-lightning torchmetrics

In [ ]:
import torch
import torchvision
from torchvision import transforms
import torchmetrics
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint
from pytorch_lightning.loggers import TensorBoardLogger

In [ ]:
labels = pd.read_csv("/content/stage_2_train_labels.csv")

In [ ]:
labels.head(6)

In [ ]:
labels = labels.drop_duplicates('patientId')

In [ ]:
labels.head(6)

In [ ]:
ROOT_PATH = Path("/content/stage_2_train_images/")
SAVE_PATH = Path("Processed/")

In [ ]:
fig,axis = plt.subplots(3,3, figsize=(9,9))
c = 0

for i in range(3):
  for j in range(3):
    patient_id = labels.patientId.iloc[c]
    dcm_path = ROOT_PATH/patient_id
    dcm_path = dcm_path.with_suffix('.dcm')
    dcm = pydicom.dcmread(dcm_path).pixel_array
    label = labels.Target.iloc[c]



    axis[i][j].imshow(dcm,cmap='gray')
    axis[i][j].set_title(label)
    c+=1

In [ ]:
sums = 0
sums_squared = 0

for c,patient_id in enumerate(tqdm(labels.patientId)):
  dcm_path = ROOT_PATH/patient_id
  dcm_path = dcm_path.with_suffix('.dcm')
  dcm = pydicom.dcmread(dcm_path).pixel_array / 255
  label = labels.Target.iloc[c]

  dcm_array = cv2.resize(dcm,(224,224)).astype(np.float16)

  train_or_val = "train" if c < 24000 else "val"

  current_save_path = SAVE_PATH/train_or_val/str(label)
  current_save_path.mkdir(parents=True,exist_ok=True)
  np.save(current_save_path/patient_id,dcm_array)

  normalizer = dcm_array.shape[0] * dcm_array.shape[1]
  if train_or_val == "train":
    sums+= np.sum(dcm_array) / normalizer
    sums_squared += (np.power(dcm_array,2).sum()) / normalizer

mean = sums / 24000
std =np.sqrt(sums_squared / 24000 - (mean**2))



In [ ]:
print(f"Mean: {mean}")
print(f"STD: {std}")

In [ ]:
def load_file(path):
  return np.load(load_file).astype(np.float32)

In [ ]:
train_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(0.0853,0.2340),
    transforms.RandomAffine(degrees=(-5,5),translate=(0,0.05),scale=(0.9,1.1)),
    transforms.RandomResizedCrop(size=(224,224),scale=(0.35,1))
])

val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(0.0853,0.2340)
])

In [ ]:
train_data = torchvision.datasets.DatasetFolder("Processed/train",loader=load_file,extensions="npy",transform=train_transform)
val_data = torchvision.datasets.DatasetFolder("Processed/val",loader=load_file,extensions="npy",transform=val_transform)

In [ ]:
train_loader = torch.utils.data.DataLoader(train_data,batch_size=64,num_workers=4,shuffle=True)
val_loader = torch.utils.data.DataLoader(val_data,batch_size=64,num_workers=4,shuffle=True)

In [ ]:
torchvision.models.resnet18()

In [ ]:
class PneumoniaModel(pl.LightningModule):
    def __init__(self):
      super().__init__()

      self.model = torchvision.models.resnet18()
      self.model.conv1 = torch.nn.Conv2d(1,64,kernel_size=(7,7),stride=(2,2),padding=(3,3),bias=False)
      self.model.fc = torch.nn.Linear(in_features=512,out_features=1,bias=True)

      self.loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor([3.0]))

      self.train_acc = torchmetrics.Accuracy(task="binary")
      self.val_acc = torchmetrics.Accuracy(task="binary")

    def forward(self,data):
      pred = self.model(data)
      return pred

    def training_step(self,batch,batch_idx):
      x_ray,label=batch
      label = label.float()
      pred = self(x_ray)[:,0]
      loss = self.loss_fn(pred,label)

      self.log("Train loss",loss,prog_bar=True)
      acc = self.train_acc(torch.sigmoid(pred),label.int())
      self.log("Step train acc",acc,prog_bar=True)

      return loss

    def on_train_epoch_end(self,outputs):
      self.log("Train ACC",self.train_acc.compute())
      self.train_acc.reset()

    def validation_step(self,batch,batch_idx):
      x_ray,label = batch
      label = label.float()
      pred = self(x_ray)[:,0]
      loss = self.loss_fn(pred,label)

      self.log("Val loss",loss,prog_bar=True)
      acc = self.val_acc(torch.sigmoid(pred),label.int())
      self.log("Step val acc",acc,prog_bar=True)

    def on_validation_epoch_end(self,outputs):
      self.log("Val ACC", self.val_acc.compute())
      self.val_acc.reset()

    def configure_optimizers(self):
      optimizer = torch.optim.Adam(self.model.parameters(),lr=1e-4)
      return optimizer


In [ ]:
model = PneumoniaModel()
model

In [40]:
checkpoint_callback = ModelCheckpoint(
    monitor="Val ACC",
    save_top_k=1,
    mode="max",
    filename="best-model"
)

In [ ]:
trainer = pl.Trainer(
    accelerator="gpu",
    devices=1,
    logger=TensorBoardLogger(save_dir="./logs"),
    log_every_n_steps=1,
    callbacks=[checkpoint_callback],
    max_epochs=35
)